# Notebook 07 — Uncertainty (Temporal Instability) + Explainability (Clinician-facing)
**Project 3: Early Warning System for Sepsis (PhysioNet 2019)**

This notebook completes the original PhD-level promise:

- **RQ2: Stability over time**  
  Quantify risk *instability* (volatility / flip rate) and test a **stability-gated alert policy**.

- **RQ3: Explainability over time**  
  Provide **global** and **local** explanations that can be shown alongside alerts.

Outputs (saved to `results/`):
- `instability_metrics.parquet`
- `policyC_stability_sweep.csv`
- `policyC_selected.json`
- `explainability_global_perm_importance.csv`
- `explainability_local_alert_examples.csv`

> This notebook uses saved artifacts (OOF + calibration). It does not depend on raw PhysioNet files.


In [1]:

from pathlib import Path
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
from sklearn.inspection import permutation_importance
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

pd.set_option("display.max_columns", 200)

PROJECT_ROOT = Path(".").resolve()
RESULTS_DIR = PROJECT_ROOT / "results"
FIG_DIR = PROJECT_ROOT / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

OOF_CAL_PATH = RESULTS_DIR / "oof_with_calibration.parquet"
FEATURES_PATH = RESULTS_DIR / "features_02.parquet"   # if you keep features in results/
if not FEATURES_PATH.exists():
    FEATURES_PATH = PROJECT_ROOT / "features_02.parquet"  # fallback if stored at root

FINAL_DECISION_PATH = RESULTS_DIR / "final_decision.json"
UTILITY_PATH = RESULTS_DIR / "policyA_selected_utility.json"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("OOF_CAL_PATH exists:", OOF_CAL_PATH.exists())
print("FEATURES_PATH:", FEATURES_PATH, "| exists:", FEATURES_PATH.exists())
print("FINAL_DECISION exists:", FINAL_DECISION_PATH.exists())


PROJECT_ROOT: C:\Users\Admin\anaconda_projects\f41dcae4-d989-44e3-8a04-984a86780085\notebooks
RESULTS_DIR: C:\Users\Admin\anaconda_projects\f41dcae4-d989-44e3-8a04-984a86780085\notebooks\results
OOF_CAL_PATH exists: True
FEATURES_PATH: C:\Users\Admin\anaconda_projects\f41dcae4-d989-44e3-8a04-984a86780085\notebooks\results\features_02.parquet | exists: True
FINAL_DECISION exists: True


## 1) Load OOF + choose operating τ for instability analysis
We use **calibrated** probabilities `p_cal`.  
τ is taken from `final_decision.json` if available, otherwise defaults to 0.08544.


In [2]:

assert OOF_CAL_PATH.exists(), f"Missing {OOF_CAL_PATH}. Run Notebook 04 first."

oof = pd.read_parquet(OOF_CAL_PATH).sort_values(["patient_id","ICULOS"]).reset_index(drop=True)

req = {"patient_id","ICULOS","y","p_cal"}
missing = req - set(oof.columns)
assert not missing, f"OOF missing columns: {missing}"

tau_default = 0.08544
tau = tau_default
if FINAL_DECISION_PATH.exists():
    final = json.loads(FINAL_DECISION_PATH.read_text())
    tau = float(final.get("tau", tau_default))
elif (RESULTS_DIR / "final_operating_point.csv").exists():
    tau = float(pd.read_csv(RESULTS_DIR / "final_operating_point.csv")["tau"].iloc[0])

print("Using tau for instability analysis:", tau)

y = oof["y"].astype(int).values
p = oof["p_cal"].astype(float).values
print("Row-level OOF AUROC:", roc_auc_score(y,p))
print("Row-level OOF AUPRC:", average_precision_score(y,p))
print("Brier:", brier_score_loss(y,p))


Using tau for instability analysis: 0.0810526315789473
Row-level OOF AUROC: 0.9343177905793643
Row-level OOF AUPRC: 0.1190227416401703
Brier: 0.010116269139269314


## 2) Temporal instability metrics (per patient-hour)

We quantify 3 stability signals on the risk trajectory `p_cal`:

- `risk_std_w`: rolling std over last **w hours**  
- `risk_mad_w`: rolling mean absolute delta (`|p_t - p_{t-1}|`) over last **w hours**  
- `flip_rate_w`: fraction of sign flips around τ within last **w hours** (how often risk crosses the decision boundary)


In [3]:

def add_instability_metrics(df: pd.DataFrame, tau: float, w: int = 6) -> pd.DataFrame:
    g = df.sort_values(["patient_id","ICULOS"]).copy()

    g["risk_delta"] = g.groupby("patient_id")["p_cal"].diff().abs()
    g["risk_std_w"] = g.groupby("patient_id")["p_cal"].rolling(w, min_periods=2).std().reset_index(level=0, drop=True)
    g["risk_mad_w"] = g.groupby("patient_id")["risk_delta"].rolling(w, min_periods=2).mean().reset_index(level=0, drop=True)

    sign = np.sign(g["p_cal"].values - tau)
    g["sign"] = sign
    g["sign_prev"] = g.groupby("patient_id")["sign"].shift(1)
    g["flip"] = ((g["sign"] * g["sign_prev"]) < 0).astype(float)
    g["flip_rate_w"] = g.groupby("patient_id")["flip"].rolling(w, min_periods=2).mean().reset_index(level=0, drop=True)

    g.drop(columns=["sign_prev"], inplace=True)
    return g

W = 6
oof_instab = add_instability_metrics(oof, tau=tau, w=W)

out_instab = RESULTS_DIR / "instability_metrics.parquet"
oof_instab.to_parquet(out_instab, index=False)
print("Saved:", out_instab)

oof_instab[["p_cal","risk_std_w","risk_mad_w","flip_rate_w"]].describe().T


Saved: C:\Users\Admin\anaconda_projects\f41dcae4-d989-44e3-8a04-984a86780085\notebooks\results\instability_metrics.parquet


,count,mean,std,min,25%,50%,75%,max
p_cal,19145.0,0.011241,0.033395,0.0,0.0,0.0,0.001528,0.600000
risk_std_w,18652.0,0.001924,0.012504,0.0,0.0,0.0,0.000239,0.363978
risk_mad_w,18161.0,0.001343,0.010249,0.0,0.0,0.0,0.000255,0.495765
flip_rate_w,18652.0,0.011992,0.062757,0.0,0.0,0.0,0.000000,0.833333


## 3) Stability-gated alert policy (Policy C)

**Policy C:** Alert only if `p_cal >= τ` AND risk is stable: `risk_std_w <= s_max`  
Then apply a lockout (6 hours).


In [4]:

has_onset_cols = {"sepsis_any","event_iculos"}.issubset(oof_instab.columns)
print("Has onset columns in OOF:", has_onset_cols)

LOCKOUT_HOURS = 6

def first_crossing_with_stability(df_patient: pd.DataFrame, tau: float, s_max: float, lockout_hours: int = 6):
    g = df_patient.sort_values("ICULOS")
    alert_times = []
    next_allowed = -np.inf
    for t, p, s in zip(g["ICULOS"].values.astype(float),
                      g["p_cal"].values.astype(float),
                      g["risk_std_w"].values.astype(float)):
        if t < next_allowed:
            continue
        if (p >= tau) and (not np.isnan(s)) and (s <= s_max):
            alert_times.append(float(t))
            next_allowed = float(t) + lockout_hours
    return alert_times

def evaluate_policyC(df_all: pd.DataFrame, tau: float, s_max: float, lockout_hours: int = 6):
    rows = []
    for pid, g in df_all.groupby("patient_id", sort=False):
        sepsis_any = int(g["sepsis_any"].iloc[0])
        onset = g["event_iculos"].iloc[0]

        alerts = first_crossing_with_stability(g, tau=tau, s_max=s_max, lockout_hours=lockout_hours)

        rows.append({
            "patient_id": pid,
            "sepsis_any": sepsis_any,
            "onset": onset,
            "n_alerts": len(alerts),
            "first_alert": alerts[0] if len(alerts) else np.nan,
            "patient_hours": len(g)
        })
    per = pd.DataFrame(rows)

    alerts_per_100h = 100.0 * per["n_alerts"].sum() / max(per["patient_hours"].sum(), 1)

    out = {
        "tau": float(tau),
        "s_max": float(s_max),
        "alerts_per_100_patient_hours": float(alerts_per_100h),
    }

    sepsis = per[per["sepsis_any"] == 1].copy()
    nonsepsis = per[per["sepsis_any"] == 0].copy()

    out["nonsepsis_alert_rate"] = float((nonsepsis["n_alerts"] > 0).mean()) if len(nonsepsis) else np.nan

    det = 0
    leads = []
    for _, r in sepsis.iterrows():
        if pd.isna(r["first_alert"]) or pd.isna(r["onset"]):
            continue
        if r["first_alert"] < r["onset"]:
            det += 1
            leads.append(float(r["onset"]) - float(r["first_alert"]))
    out["sepsis_detection_rate"] = float(det / len(sepsis)) if len(sepsis) else np.nan
    out["lead_time_median"] = float(np.median(leads)) if len(leads) else np.nan
    out["n_sepsis"] = int(len(sepsis))
    out["n_nonsepsis"] = int(len(nonsepsis))

    return out

if not has_onset_cols:
    raise ValueError(
        "oof_with_calibration.parquet must include sepsis_any and event_iculos for lead-time evaluation. "
        "Regenerate it in Notebook 04 by merging patient_events before saving."
    )

std_series = oof_instab["risk_std_w"].dropna()
quantiles = np.unique(np.quantile(std_series, np.linspace(0.05, 0.95, 25)))
quantiles = [float(x) for x in quantiles if np.isfinite(x)]

sweep_rows = [evaluate_policyC(oof_instab, tau=tau, s_max=s, lockout_hours=LOCKOUT_HOURS) for s in quantiles]
sweepC = pd.DataFrame(sweep_rows).sort_values("s_max").reset_index(drop=True)

out_sweep = RESULTS_DIR / "policyC_stability_sweep.csv"
sweepC.to_csv(out_sweep, index=False)
print("Saved:", out_sweep)

sweepC.head()


Has onset columns in OOF: True
Saved: C:\Users\Admin\anaconda_projects\f41dcae4-d989-44e3-8a04-984a86780085\notebooks\results\policyC_stability_sweep.csv


,tau,s_max,alerts_per_100_patient_hours,nonsepsis_alert_rate,sepsis_detection_rate,lead_time_median,n_sepsis,n_nonsepsis
0,0.081053,0.000000,0.621572,0.0,0.692308,57.0,39,454
1,0.081053,0.000178,0.621572,0.0,0.692308,57.0,39,454
2,0.081053,0.000624,0.746931,0.0,0.692308,57.0,39,454
3,0.081053,0.000624,0.746931,0.0,0.692308,57.0,39,454
4,0.081053,0.000789,0.746931,0.0,0.692308,57.0,39,454


## 4) Select `s_max` under guardrails
Default guardrails:
- `nonsepsis_alert_rate <= 0.10`
- `alerts_per_100_patient_hours <= 1.0`

Objective:
- maximize **detection**, tie-break by **median lead**, then minimize burden.


In [5]:

MAX_FALSE_ALERT = 0.10
MAX_ALERTS_PER_100H = 1.0

feasible = sweepC[
    (sweepC["nonsepsis_alert_rate"] <= MAX_FALSE_ALERT) &
    (sweepC["alerts_per_100_patient_hours"] <= MAX_ALERTS_PER_100H)
].copy()

cand = feasible if not feasible.empty else sweepC.copy()

cand = cand.sort_values(
    ["sepsis_detection_rate", "lead_time_median", "alerts_per_100_patient_hours"],
    ascending=[False, False, True]
)

best = cand.iloc[0].to_dict()
print("Selected Policy C:", best)

policyC = {
    "policy": "C_stability_gated_first_crossing",
    "tau": float(best["tau"]),
    "s_max": float(best["s_max"]),
    "window_hours": int(W),
    "lockout_hours": int(LOCKOUT_HOURS),
    "guardrails": {"max_false_alert_patients": MAX_FALSE_ALERT, "max_alerts_per_100h": MAX_ALERTS_PER_100H},
    "metrics": {k: best.get(k) for k in ["sepsis_detection_rate","nonsepsis_alert_rate","alerts_per_100_patient_hours","lead_time_median"]}
}

out_sel = RESULTS_DIR / "policyC_selected.json"
out_sel.write_text(json.dumps(policyC, indent=2))
print("Saved:", out_sel)


Selected Policy C: {'tau': 0.0810526315789473, 's_max': 0.0, 'alerts_per_100_patient_hours': 0.6215722120658135, 'nonsepsis_alert_rate': 0.0, 'sepsis_detection_rate': 0.6923076923076923, 'lead_time_median': 57.0, 'n_sepsis': 39.0, 'n_nonsepsis': 454.0}
Saved: C:\Users\Admin\anaconda_projects\f41dcae4-d989-44e3-8a04-984a86780085\notebooks\results\policyC_selected.json


## 5) Explainability (global + local)

Global: permutation importance for HGB (AUPRC).  
Local: clinician-facing top feature contributions at alert time using a standardized Logistic Regression surrogate.


In [6]:

assert FEATURES_PATH.exists(), (
    f"Missing features file at {FEATURES_PATH}. "
    "Place features_02b.parquet in repo root or results/."
)

feat = pd.read_parquet(FEATURES_PATH)

id_cols = ["patient_id","ICULOS"]
target_candidates = [c for c in ["y_within_H","y","target"] if c in feat.columns]
assert all(c in feat.columns for c in id_cols), f"features file must contain {id_cols}"
assert len(target_candidates) >= 1, "Could not find target column (y_within_H or y) in features file."
target_col = target_candidates[0]

# Align with OOF cohort by key intersection
feat = feat.merge(oof[["patient_id","ICULOS"]].drop_duplicates(), on=id_cols, how="inner")

y = feat[target_col].astype(int).values
groups = feat["patient_id"].values

drop_cols = set(id_cols + [target_col])
X = feat[[c for c in feat.columns if c not in drop_cols]]
num_cols = list(X.columns)

print("Explainability data:", X.shape, "| prevalence:", y.mean())

preprocess = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num_cols),
], remainder="drop")

hgb = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=300, random_state=42)
hgb_pipe = Pipeline([("prep", preprocess), ("clf", hgb)])

logreg_pipe = Pipeline([
    ("prep", ColumnTransformer([("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                                                 ("scaler", StandardScaler())]), num_cols)])),
    ("clf", LogisticRegression(penalty="l2", C=1.0, max_iter=2000, solver="lbfgs"))
])


Explainability data: (19145, 624) | prevalence: 0.011177853225385219


In [7]:

# Global permutation importance (average over folds)
gkf = GroupKFold(n_splits=5)
imps = []
fold = 0
for tr, te in gkf.split(X, y, groups=groups):
    fold += 1
    hgb_pipe.fit(X.iloc[tr], y[tr])
    r = permutation_importance(
        hgb_pipe, X.iloc[te], y[te],
        scoring="average_precision",
        n_repeats=5,
        random_state=fold
    )
    imps.append(pd.DataFrame({
        "feature": num_cols,
        "importance_mean": r.importances_mean,
        "importance_std": r.importances_std,
        "fold": fold
    }))
    print("Fold", fold, "done")

imp_all = pd.concat(imps, ignore_index=True)
imp_global = (imp_all.groupby("feature")[["importance_mean","importance_std"]]
              .mean()
              .sort_values("importance_mean", ascending=False)
              .reset_index())

out_imp = RESULTS_DIR / "explainability_global_perm_importance.csv"
imp_global.to_csv(out_imp, index=False)
print("Saved:", out_imp)
imp_global.head(20)


C:\Users\Admin\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\Admin\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Users\Admin\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Admin\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^

Fold 1 done
Fold 2 done
Fold 3 done
Fold 4 done
Fold 5 done
Saved: C:\Users\Admin\anaconda_projects\f41dcae4-d989-44e3-8a04-984a86780085\notebooks\results\explainability_global_perm_importance.csv


,feature,importance_mean,importance_std
0,pH__std_24h,0.004426,0.001290
1,Glucose__max_12h,0.003222,0.001116
2,MAP__std_6h,0.002688,0.000167
3,FiO2__max_24h,0.002671,0.000192
4,BUN__mean_24h,0.002647,0.000385
5,Temp__min_12h,0.002541,0.000391
6,Platelets__mean_24h,0.002382,0.000400
7,Temp__mean_6h,0.002316,0.000306
8,Fibrinogen__min_24h,0.002204,0.000382
9,PaCO2__prev1,0.002160,0.000387


In [8]:

# Local explanations: train surrogate on full data
logreg_pipe.fit(X, y)

coef = logreg_pipe.named_steps["clf"].coef_.ravel()
coef_df = pd.DataFrame({"feature": num_cols, "coef": coef}).sort_values("coef", ascending=False)

print("Top positive drivers:")
display(coef_df.head(12))
print("Top negative drivers:")
display(coef_df.tail(12))


Top positive drivers:


,feature,coef
437,Glucose__max_24h,2.139687
480,Fibrinogen__min_24h,1.946648
3,SBP__miss,1.508922
229,BaseExcess__max_12h,1.342777
392,FiO2__min_24h,1.295820
475,WBC__mean_24h,1.124651
352,HR__min_24h,1.095112
204,Temp__min_12h,1.064645
421,Calcium__max_24h,1.060091
368,MAP__min_24h,1.058486


Top negative drivers:


,feature,coef
456,Bilirubin_total__min_24h,-1.093927
372,DBP__min_24h,-1.115384
391,FiO2__mean_24h,-1.136461
351,HR__mean_24h,-1.139512
4,MAP__miss,-1.172920
438,Glucose__std_24h,-1.264570
478,WBC__std_24h,-1.392534
310,Hct__std_12h,-1.482857
482,Fibrinogen__std_24h,-1.513630
5,DBP__miss,-1.650633


In [9]:

# Build alert examples from Policy A using OOF trajectory
def policy_first_crossing_lockout(df_patient, prob_col, tau, lockout_hours=6):
    g = df_patient.sort_values("ICULOS")
    alert_times = []
    next_allowed = -np.inf
    for t, prob in zip(g["ICULOS"].values.astype(float), g[prob_col].values.astype(float)):
        if t < next_allowed:
            continue
        if prob >= tau:
            alert_times.append(float(t))
            next_allowed = float(t) + lockout_hours
    return alert_times

tau_A = tau
lockout = 6

alerts = []
for pid, g in oof_instab.groupby("patient_id", sort=False):
    sepsis_any = int(g["sepsis_any"].iloc[0])
    onset = g["event_iculos"].iloc[0]
    times = policy_first_crossing_lockout(g, "p_cal", tau_A, lockout_hours=lockout)
    if not times:
        continue
    alerts.append({"patient_id": pid, "sepsis_any": sepsis_any, "onset": onset, "alert_time": int(times[0])})

alerts = pd.DataFrame(alerts)
print("Patients with ≥1 Policy A alert:", len(alerts))

N = 5
examples = pd.concat([
    alerts[alerts["sepsis_any"]==1].head(N),
    alerts[alerts["sepsis_any"]==0].head(N),
], ignore_index=True)

examples


Patients with ≥1 Policy A alert: 39


,patient_id,sepsis_any,onset,alert_time
0,p000890,1,73.0,2
1,p001669,1,6.0,2
2,p002332,1,136.0,122
3,p005186,1,105.0,19
4,p006461,1,108.0,2


In [10]:

# Local contribution table at alert time:
# contribution_j = coef_j * z_j  where z is standardized feature value at that time.

feat_keyed = feat.set_index(["patient_id","ICULOS"])
rows = []

for _, ex in examples.iterrows():
    pid = ex["patient_id"]
    t = int(ex["alert_time"])
    if (pid, t) not in feat_keyed.index:
        continue

    x_row = feat_keyed.loc[(pid, t), num_cols]
    X_row_df = pd.DataFrame([x_row.values], columns=num_cols)

    Xt = logreg_pipe.named_steps["prep"].transform(X_row_df)  # standardized
    contrib = Xt.ravel() * coef

    topk = 10
    idx = np.argsort(np.abs(contrib))[::-1][:topk]
    for j in idx:
        rows.append({
            "patient_id": pid,
            "alert_time": t,
            "sepsis_any": int(ex["sepsis_any"]),
            "onset": ex["onset"],
            "feature": num_cols[j],
            "contribution": float(contrib[j]),
            "z_value": float(Xt.ravel()[j]),
            "coef": float(coef[j]),
            "raw_value": float(x_row.iloc[j])
        })

local = pd.DataFrame(rows).sort_values(["patient_id","alert_time","sepsis_any","contribution"], ascending=[True,True,False,False])

out_local = RESULTS_DIR / "explainability_local_alert_examples.csv"
local.to_csv(out_local, index=False)
print("Saved:", out_local)

local.head(40)


Saved: C:\Users\Admin\anaconda_projects\f41dcae4-d989-44e3-8a04-984a86780085\notebooks\results\explainability_local_alert_examples.csv


,patient_id,alert_time,sepsis_any,onset,feature,contribution,z_value,coef,raw_value
0,p000890,2,1,73.0,SBP__miss,4.122354,2.731986,1.508922,1.000000
2,p000890,2,1,73.0,Hct__std_12h,0.741929,-0.500337,-1.482857,NaN
3,p000890,2,1,73.0,WBC__std_24h,0.604868,-0.434365,-1.392534,NaN
4,p000890,2,1,73.0,Glucose__std_12h,0.519603,-0.273028,-1.903113,NaN
5,p000890,2,1,73.0,Glucose__min_24h,0.491927,-0.163280,-3.012781,NaN
7,p000890,2,1,73.0,MAP__miss,0.390803,-0.333188,-1.172920,0.000000
9,p000890,2,1,73.0,Calcium__std_24h,0.333730,-0.333332,-1.001195,NaN
8,p000890,2,1,73.0,Platelets__std_24h,-0.387278,-0.392858,0.985795,NaN
6,p000890,2,1,73.0,Glucose__max_24h,-0.433379,-0.202543,2.139687,NaN
1,p000890,2,1,73.0,DBP__miss,-2.540483,1.539097,-1.650633,1.000000


## 6) Final notes
- Policy C shows how **instability** can gate alerts (reduce “jitter alerts”).
- The surrogate explanations provide a **clinician-facing** story: top drivers at the alert time.
